# 02 — Personal Sequence Construction (Phase 2)

Thin wrapper over `src/personal.py`. Two phased haplotype windows per (gene, individual) via batched
genotype streaming; sequences built on the fly (not materialized). Full record:
`docs/PHASE2_personal_sequences.md`. Validation + variant stats: `results/phase2_variant_stats.csv`.

In [ ]:
import sys, subprocess; sys.path.insert(0, '../src')
# validate batched==single path, characterize cis-variation, measure throughput (chr22)
subprocess.run([sys.executable, '../scripts/phase2_validate_sequences.py'], check=True)

In [ ]:
import pandas as pd
df = pd.read_csv('../results/phase2_variant_stats.csv')
print('per-window personal cis-variation (chr22 sample):')
df[['n_snv','n_segregating','mean_alt_per_indiv','mean_het_per_indiv']].describe().round(1)

## Build one individual's two haplotypes (demonstration)

In [ ]:
from paths import REF_DIR, PROCESSED, CACHE, kgp_vcf_url
from genome import Reference
import personal, pandas as pd
ref = Reference(REF_DIR/'Homo_sapiens.GRCh38.dna.chromosome.22.fa')
gm = pd.read_parquet(PROCESSED/'gene_meta.parquet'); im = pd.read_parquet(PROCESSED/'individual_meta.parquet')
samples = list(im.index); sf = personal.write_sample_file(samples, CACHE/'geuvadis_449_samples.txt')
g = gm[gm.chrom=='chr22'].iloc[0]
wg = personal.stream_window_genotypes(ref, kgp_vcf_url('chr22'), 'chr22', int(g.tss), samples, sf, strand=g.strand)
h1, h2 = personal.haplotypes(wg, samples[0])
print(g.gene_name, '| variants in window:', wg.n_variants, '| hap lens:', len(h1), len(h2))
print('stats:', personal.variant_stats(wg))